# Révision — Machine Learning (TP et TD)

Ce notebook rassemble les notions travaillées dans les TP/TD : exploration des données, préparation d'un jeu de données, régression, arbres de décision, évaluation, surapprentissage, descente de gradient et PyTorch.

**Fil conducteur :** à partir de données, définir une cible et des variables explicatives, entraîner un modèle sans fuite d'information, mesurer sa capacité à généraliser et comprendre comment ses paramètres sont appris.

**Données utilisées en cours :** prix de maisons de l'Iowa (`R5.11/data/iowa_data.csv`) et manchots Palmer (dataset `penguins` de seaborn). Les rappels et exemples ci-dessous ne nécessitent pas de charger ces jeux de données.

## 1. Vocabulaire et démarche générale

- **Observation / exemple** : une ligne du tableau (par exemple une maison).
- **Variable explicative / feature** : information utilisée pour prédire, notée `X` (surface, année de construction…).
- **Cible / target** : valeur à prédire, notée `y` ou `t` (par exemple `SalePrice`).
- **Modèle** : fonction paramétrée qui transforme `X` en prédiction `y_pred`.
- **Entraînement (fit)** : choix des paramètres du modèle à partir des données d'entraînement.
- **Prédiction (predict)** : application du modèle à de nouvelles observations.
- **Métrique / perte** : mesure numérique de l'écart entre les prédictions et les valeurs réelles.

Démarche type : explorer → nettoyer/préparer → séparer les données → entraîner → prédire → évaluer → comparer/ajuster le modèle.

## 2. Charger et explorer les données (EDA)

Un DataFrame pandas permet d'inspecter les dimensions, les types, les statistiques et les valeurs manquantes avant de choisir les variables. `describe()` donne notamment le nombre de valeurs (`count`), la moyenne, l'écart-type, les quartiles et les extrêmes.

In [ ]:
import pandas as pd

# Exemple pour les données Iowa, depuis le dossier R5.11.
data = pd.read_csv("data/iowa_data.csv", index_col="Id")
print("Dimensions :", data.shape)
display(data.head())
display(data.dtypes)
display(data.describe())
print("Valeurs manquantes par colonne :\n", data.isna().sum().sort_values(ascending=False).head())

### À retenir pour l'exploration

- `data.shape` renvoie `(nombre_de_lignes, nombre_de_colonnes)`.
- `data.dtypes` montre la nature des colonnes ; `data.describe()` résume les colonnes numériques.
- `data.isna().sum()` compte les valeurs manquantes par colonne.
- Un histogramme aide à voir la distribution d'une variable ; un nuage de points `scatter(x, y)` aide à repérer une relation entre une feature et la cible.
- Il faut comprendre le sens et les unités des variables avant de construire le modèle.

Pour la cible, on peut supprimer les lignes où **la cible** manque. Pour les features, il faut choisir une stratégie adaptée (suppression, imputation, encodage des catégories) plutôt que supposer que toutes les données sont complètes.

## 3. Construire les données d'entraînement et d'évaluation

Séparer `X` (features) de `y` (cible), puis séparer les observations. Le modèle apprend sur `X_train, y_train` ; les performances se mesurent sur des observations qui n'ont pas servi à l'entraînement. `random_state` rend le découpage reproductible.

Dans les TP, le jeu restant est appelé validation. En pratique, on distingue souvent **train** (apprentissage), **validation** (choix des hyperparamètres) et **test** (évaluation finale, utilisé une seule fois). Ne jamais entraîner sur le jeu d'évaluation.

In [ ]:
from sklearn.model_selection import train_test_split

target_name = "SalePrice"
feature_names = ["LotArea", "YearBuilt", "1stFlrSF", "2ndFlrSF",
                 "FullBath", "BedroomAbvGr", "TotRmsAbvGrd"]

data_clean = data.dropna(subset=[target_name]).copy()
X = data_clean[feature_names]
y = data_clean[target_name]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.5, random_state=42
)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

## 4. Régression et métrique MAE

La régression prédit une valeur numérique. L'erreur absolue moyenne (**MAE**) est

$$\mathrm{MAE}(y,\hat y)=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

Elle s'exprime dans la même unité que la cible : une MAE de 20 000 signifie une erreur absolue moyenne d'environ 20 000 unités monétaires. Plus bas est généralement meilleur, à condition de comparer sur les mêmes données.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
val_predictions = tree.predict(X_val)
val_mae = mean_absolute_error(y_val, val_predictions)
print(f"MAE validation : {val_mae:,.0f}")

# La MAE se recalcule aussi directement :
mae_manuel = (y_val - val_predictions).abs().mean()
print(f"Vérification : {mae_manuel:,.0f}")

## 5. Arbres de décision : profondeur, underfitting et overfitting

Un arbre de régression découpe l'espace des features en régions et prédit une valeur (typiquement une moyenne) dans chacune. Avec une seule feature, sa prédiction forme un escalier.

- **Profondeur trop faible** : modèle trop simple, qui ne capture pas la structure (**underfitting**). Erreur d'entraînement et de validation élevées.
- **Profondeur très élevée** : arbre complexe qui peut mémoriser les exemples (**overfitting**). Erreur d'entraînement faible, mais erreur de validation plus grande.
- On choisit `max_depth` en comparant les performances de validation ; le minimum de l'erreur de validation indique une profondeur intéressante pour ce découpage.

La profondeur est un **hyperparamètre** : on la fixe avant/après essais, elle n'est pas directement apprise comme les paramètres d'un modèle linéaire. Garder un jeu de test séparé évite de choisir le modèle sur le test.

In [ ]:
depths = range(1, 21)
train_errors, val_errors = [], []

for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_errors.append(mean_absolute_error(y_train, model.predict(X_train)))
    val_errors.append(mean_absolute_error(y_val, model.predict(X_val)))

import matplotlib.pyplot as plt
plt.plot(depths, train_errors, label="Entraînement")
plt.plot(depths, val_errors, label="Validation")
plt.xlabel("Profondeur maximale")
plt.ylabel("MAE")
plt.legend()
plt.show()

## 6. Régression linéaire : modèle, vecteurs et MSE

Avec une variable explicative, le modèle est une droite :

$$f_w(x)=w_0+w_1x.$$

`w0` est l'ordonnée à l'origine et `w1` la pente. En ajoutant une feature constante `x0 = 1`, on écrit `x_aug = (1, x)` et `w = (w0, w1)`, donc la prédiction est le produit scalaire $w^T x_{aug}$. Pour `N` observations, la matrice `X_aug` a `N` lignes et deux colonnes : des 1 puis les valeurs de `x`.

La moyenne des erreurs quadratiques (**MSE**) est

$$E(w)=\frac{1}{N}\sum_{i=1}^{N}(f_w(x_i)-t_i)^2.$$

Le TD utilise cette convention. Pour dériver les règles de mise à jour, le TD4/TD5 écrit aussi $E(w)=\frac{1}{2N}\sum_i e_i^2$ : le facteur $1/2$ simplifie la dérivée. Ces deux pertes ont le même minimum ; leurs gradients, et donc le pas d'apprentissage effectif, diffèrent d'un facteur 2.

In [ ]:
import numpy as np

# Exemple illustratif : x = feature, t = valeur cible
x = np.array([1.0, 2.0, 3.0, 4.0])
t = np.array([2.0, 3.0, 5.0, 4.0])
w = np.array([0.5, 1.0])  # w0, w1
X_aug = np.column_stack([np.ones_like(x), x])
y_pred = X_aug @ w
mse = np.mean((y_pred - t) ** 2)
print("Prédictions :", y_pred)
print("MSE :", mse)

## 7. Minimum analytique et solution OLS

Pour une pente `w1` fixée, annuler la dérivée par rapport à `w0` donne

$$w_0^*=\bar t-w_1\bar x.$$

Pour optimiser tous les poids simultanément, en notation matricielle, les équations normales sont

$$X^T Xw=X^Tt.$$

La solution OLS (*ordinary least squares*) est formellement $(X^TX)^{-1}X^Tt$. En code, préférer `np.linalg.solve(X.T @ X, X.T @ t)` à calculer explicitement l'inverse : c'est en général plus stable et efficace. `sklearn.linear_model.LinearRegression` fournit une implémentation pratique de la régression linéaire.

Une grille de paramètres et une carte de niveaux aident à visualiser la perte, mais leur minimum est seulement approximatif et dépend de la résolution de la grille.

In [ ]:
X_aug = np.column_stack([np.ones_like(x), x])
w_ols = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ t)
w_inv = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ t
print("w0, w1 (solve) :", w_ols)
print("Solutions proches ?", np.allclose(w_ols, w_inv))

from sklearn.linear_model import LinearRegression
linear_model = LinearRegression().fit(x.reshape(-1, 1), t)
print("Intercept, coefficient :", linear_model.intercept_, linear_model.coef_[0])

## 8. Descente de gradient

Quand le nombre de paramètres devient grand, la solution analytique par inversion de matrice est coûteuse. La descente de gradient améliore les paramètres itération après itération dans la direction opposée au gradient.

Avec $e_i=w_0+w_1x_i-t_i$ et $E=\frac{1}{2N}\sum_i e_i^2$ :

$$\frac{\partial E}{\partial w_0}=\frac{1}{N}\sum_i e_i,\qquad
\frac{\partial E}{\partial w_1}=\frac{1}{N}\sum_i e_i x_i,$$

puis, pour un taux d'apprentissage $\eta$ :

$$w_0\leftarrow w_0-\eta\frac{\partial E}{\partial w_0},\qquad
w_1\leftarrow w_1-\eta\frac{\partial E}{\partial w_1}.$$

Les deux gradients doivent être calculés avec les **anciens** poids, puis les poids mis à jour. Un taux `eta` trop grand peut faire diverger/osciller la perte ; trop petit, il rend la convergence lente. La normalisation des features (et parfois de la cible) aide à choisir un pas stable. Il faut ajuster le scaler sur les données d'entraînement uniquement, puis réutiliser ce scaler.

In [ ]:
def gradient_descent(x, t, eta=0.1, n_iter=100):
    w0, w1 = 0.0, 0.0
    history = []
    n = len(x)

    for _ in range(n_iter):
        errors = w0 + w1 * x - t
        grad_w0 = np.mean(errors)
        grad_w1 = np.mean(errors * x)
        w0 -= eta * grad_w0
        w1 -= eta * grad_w1
        history.append(0.5 * np.mean(errors ** 2))

    return w0, w1, history

w0_gd, w1_gd, history = gradient_descent(x, t, eta=0.05, n_iter=500)
print("Poids appris :", w0_gd, w1_gd)
plt.plot(history)
plt.xlabel("Itération")
plt.ylabel("Perte 1/2 MSE")
plt.show()

### Standardisation

La standardisation transforme une variable selon $z=(x-\mu)/\sigma$, pour obtenir une moyenne proche de 0 et un écart-type proche de 1. Elle rend les échelles des features comparables et peut faciliter l'optimisation. Elle est particulièrement utile pour la descente de gradient ; les arbres de décision n'en ont généralement pas besoin.

Pour éviter la fuite d'information : `scaler.fit_transform(X_train)` puis `scaler.transform(X_val)`. Ne pas ajuster le scaler sur l'ensemble complet avant le découpage.

## 9. Même régression avec PyTorch

- Un **tenseur** est le conteneur de données de PyTorch.
- `requires_grad=True` demande à autograd de suivre les opérations et de calculer les dérivées.
- `loss.backward()` calcule les gradients ; ils sont stockés dans `param.grad`.
- `optimizer.step()` applique la mise à jour des paramètres.
- `optimizer.zero_grad()` est nécessaire à chaque itération : par défaut, PyTorch accumule les gradients.

Boucle standard : remettre les gradients à zéro → calculer prédiction/perte → `backward()` → `step()`. Pour examiner une valeur scalaire, utiliser `.item()`. Avec l'exemple $f(x)=x^2$ et $x=3$, autograd doit donner $dy/dx=2x=6$.

In [ ]:
import torch

x_t = torch.tensor(3.0, requires_grad=True)
y_t = x_t ** 2
y_t.backward()
print("Gradient attendu 6 :", x_t.grad.item())

In [ ]:
# Régression à une feature sur x et t, avec la perte 1/2 MSE.
# Les tenseurs d'entrée sont ici standardisés dans les TP.
x_t = torch.tensor(x, dtype=torch.float32)
t_t = torch.tensor(t, dtype=torch.float32)
w0_t = torch.zeros(1, requires_grad=True)
w1_t = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w0_t, w1_t], lr=0.05)
torch_history = []

for _ in range(500):
    optimizer.zero_grad()
    prediction = w0_t + w1_t * x_t
    loss = 0.5 * torch.mean((prediction - t_t) ** 2)
    loss.backward()
    optimizer.step()
    torch_history.append(loss.item())

print("Poids PyTorch :", w0_t.item(), w1_t.item())

## 10. Fonctions réutilisables et bonnes pratiques

Le TP1 transforme le code en petites fonctions réutilisables : nettoyage, séparation `X/y`, entraînement/évaluation. Chaque fonction doit recevoir ses dépendances en arguments au lieu de lire des variables globales cachées.

Checklist :

1. Définir clairement la cible et les features, sans inclure la cible dans `X`.
2. Nettoyer les données sans modifier involontairement l'original (`.copy()` si besoin).
3. Séparer les données avant d'ajuster tout traitement qui apprend des statistiques (scaler, imputation…).
4. Appeler `.fit(X_train, y_train)`, puis `.predict(X_val)`.
5. Calculer une métrique adaptée et interprétable.
6. Comparer erreurs d'entraînement et de validation pour repérer underfitting/overfitting.
7. Garder le test final à l'écart du choix des hyperparamètres.

## 11. Questions flash pour s'auto-tester

1. Dans une régression du prix d'une maison, que sont `X` et `y` ?
2. Quelle est la différence entre `.fit()` et `.predict()` ? Pourquoi ne pas entraîner sur la validation ?
3. Que mesure la MAE et dans quelle unité s'exprime-t-elle ?
4. Comment reconnaît-on l'overfitting sur les erreurs d'entraînement et de validation ?
5. Que représentent `w0` et `w1` dans une régression linéaire à une variable ?
6. Pourquoi la MSE avec facteur $1/2$ donne-t-elle les mêmes paramètres optimaux que la MSE sans ce facteur ?
7. Pourquoi préfère-t-on `np.linalg.solve` à l'inversion explicite pour les équations normales ?
8. Que se passe-t-il si le taux d'apprentissage est trop grand ou trop petit ?
9. Pourquoi standardiser les données avant une descente de gradient ?
10. Quel rôle jouent `requires_grad`, `.backward()`, `zero_grad()` et `optimizer.step()` ?

### Réponses condensées

1. `X` contient les caractéristiques (features) et `y` la cible à prédire.
2. `.fit()` apprend les paramètres ; `.predict()` produit les estimations. La validation doit simuler des données jamais vues.
3. La moyenne des valeurs absolues des erreurs, dans l'unité de la cible.
4. En général, erreur train basse mais validation nettement plus haute.
5. `w0` est l'ordonnée à l'origine ; `w1` est la pente.
6. Multiplier toute la fonction de perte par une constante positive ne change pas son minimiseur.
7. `solve` évite de former explicitement l'inverse et est généralement plus stable/efficace.
8. Trop grand : oscillations/divergence possibles ; trop petit : apprentissage lent.
9. Pour mettre les features à des échelles comparables et faciliter une optimisation stable.
10. Suivi des gradients ; calcul des gradients ; remise à zéro des gradients accumulés ; mise à jour des paramètres.